In [21]:
import pandas as pd
import numpy as np


In [22]:
p="new_data/"

statecodes=pd.read_csv(p+"clean_state_codes.csv")
districtcodes=pd.read_csv(p+"clean_district_codes.csv")

statepopv=(
    pd.read_csv(p+"clean_state_population.csv")
    .groupby(["state_code","state_name"],as_index=False)
    [["population","households"]]
    .sum()
)

districtpopv=(
    pd.read_csv(p+"clean_district_population.csv")
    .groupby(["state_code","district_code","district_name"],as_index=False)
    [["population","households"]]
    .sum()
)

locationv=(
    pd.read_csv(p+"clean_town_village_directory.csv")
    .groupby(
        ["state_code","district_code","subdistrict_code","location_code"],
        as_index=False
    )
    .first()
)


In [23]:
district_agents=(
    districtpopv
    [["state_code","district_code","district_name","population"]]
    .assign(
        agent_type="district",
        agent_id=lambda x: x.state_code.astype(str)+"_"+x.district_code.astype(str),
        parent_agent=lambda x: x.state_code.astype(str)
    )
    [["agent_id","agent_type","state_code","district_code","district_name","population","parent_agent"]]
)


In [24]:
state_agents=(
    statepopv
    [["state_code","state_name","population"]]
    .assign(
        agent_type="state",
        agent_id=lambda x: x.state_code.astype(str),
        parent_agent="IN"
    )
    [["agent_id","agent_type","state_code","state_name","population","parent_agent"]]
)


In [25]:
central_agent=pd.DataFrame([{
    "agent_id":"IN",
    "agent_type":"central",
    "state_code":None,
    "district_code":None,
    "name":"India",
    "population":statepopv.population.sum(),
    "parent_agent":None
}])


In [26]:
agents_df=pd.concat(
    [
        district_agents.rename(columns={"district_name":"name"}),
        state_agents.rename(columns={"state_name":"name"}),
        central_agent
    ],
    ignore_index=True
)

assert agents_df.agent_id.is_unique
assert agents_df.agent_type.isin(["district","state","central"]).all()
assert agents_df.parent_agent.isna().sum()==1


C:\Users\LATHEEF\AppData\Local\Temp\ipykernel_20104\1010760739.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  agents_df=pd.concat(


In [27]:
resources_df=pd.DataFrame([
    {
        "resource":"medical_beds",
        "category":"medical",
        "control_level":"state",
        "mobile":True,
        "priority":1
    },
    {
        "resource":"doctors",
        "category":"medical",
        "control_level":"state",
        "mobile":True,
        "priority":1
    },
    {
        "resource":"ambulances",
        "category":"medical",
        "control_level":"district",
        "mobile":True,
        "priority":1
    },
    {
        "resource":"food_packets",
        "category":"relief",
        "control_level":"central",
        "mobile":True,
        "priority":2
    },
    {
        "resource":"water_tankers",
        "category":"relief",
        "control_level":"state",
        "mobile":True,
        "priority":1
    },
    {
        "resource":"temporary_shelters",
        "category":"relief",
        "control_level":"district",
        "mobile":False,
        "priority":2
    }
])


In [28]:
assert resources_df.resource.is_unique
assert resources_df.control_level.isin(["district","state","central"]).all()
assert resources_df.priority.min()==1


In [29]:
disaster={
    "type":"flood",
    "severity":0.7,
    "affected_states":statepopv.state_code.tolist()
}


In [30]:
vulnerability_weights={
    "medical_beds":1.0,
    "doctors":0.9,
    "ambulances":1.0,
    "food_packets":0.6,
    "water_tankers":0.8,
    "temporary_shelters":0.7
}


In [31]:
demand_df=(
    district_agents
    .assign(
        severity=disaster["severity"]
    )
    .merge(resources_df,how="cross")
    .assign(
        vulnerability=lambda x: x.resource.map(vulnerability_weights),
        demand=lambda x: (
            x.population *
            x.severity *
            x.vulnerability
        ).round().astype(int)
    )
    [["agent_id","agent_type","resource","demand"]]
)


In [32]:
assert (demand_df.demand>=0).all()
assert demand_df.agent_type.eq("district").all()


In [33]:
demand_df.head()
demand_df.groupby("resource").demand.sum()


resource
ambulances            1695196965
doctors               1525677267
food_packets          1017118176
medical_beds          1695196965
temporary_shelters    1186637891
water_tankers         1356157578
Name: demand, dtype: int64

In [34]:
min_coverage_ratio=0.05


In [35]:
critical_resources=resources_df.query("priority==1").resource.tolist()


In [36]:
min_demand_df=(
    demand_df
    .query("resource in @critical_resources")
    .assign(
        min_required=lambda x: (x.demand*min_coverage_ratio).round().astype(int)
    )
    [["agent_id","resource","min_required"]]
)


In [37]:
assert (min_demand_df.min_required>=0).all()


In [38]:
sup=pd.DataFrame({
    "resource":resources_df.resource,
    "supply":[
        demand_df.loc[demand_df.resource==r,"demand"].sum()//2
        for r in resources_df.resource
    ]
})


In [40]:
a=min_demand_df.merge(sup,on="resource")
a["alloc"]=a.min_required


In [41]:
rs=(
    sup
    .merge(
        a.groupby("resource").alloc.sum().reset_index(),
        on="resource"
    )
    .assign(rem=lambda x:x.supply-x.alloc)
    [["resource","rem"]]
)


In [42]:
rd=(
    demand_df
    .merge(a[["agent_id","resource","alloc"]],on=["agent_id","resource"],how="left")
    .fillna(0)
    .assign(remd=lambda x:(x.demand-x.alloc).clip(lower=0))
)


In [45]:
pa=(
    rd
    .merge(rs,on="resource")
    .assign(
        share=lambda x:x.remd/x.groupby("resource").remd.transform("sum"),
        addv=lambda x:(x.share*x.rem).fillna(0).round().astype(int)
    )
)


In [46]:
allocdf=(
    pa
    .assign(final=lambda x:x.alloc+x.addv)
    [["agent_id","resource","final"]]
)


In [47]:
defdf=(
    demand_df
    .merge(allocdf,on=["agent_id","resource"])
    .assign(deficit=lambda x:(x.demand-x.final).clip(lower=0))
    [["agent_id","resource","deficit"]]
)


In [48]:
assert (defdf.deficit>=0).all()
assert allocdf.final.min()>=0


In [49]:
logdf=(
    allocdf
    .merge(demand_df,on=["agent_id","resource"])
    .merge(defdf,on=["agent_id","resource"])
    .assign(
        reason="ethical_minimum_then_proportional",
        status=lambda x:np.where(x.deficit>0,"partial","satisfied")
    )
    [["agent_id","resource","demand","final","deficit","status","reason"]]
)


In [50]:
logdf.head()
logdf.status.value_counts()


status
partial    2560
Name: count, dtype: int64